In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv("Hotel_Reviews.csv")

# Display first 5 rows
print(df.head())

# Display dataset information
print(df.info())

# Check dataset shape
print("Dataset Shape:", df.shape)

   Review_ID      Hotel_Name  Rating  \
0          1    City Comfort       3   
1          2       Royal Inn       2   
2          3    City Comfort       2   
3          4      Ocean View       5   
4          5  Sunrise Suites       3   

                                        Guest_Review  
0  The room was small but comfortable. The staff ...  
1  The room was small but comfortable. The staff ...  
2  The air conditioning worked perfectly. The sta...  
3  The bed was very comfortable. The housekeeping...  
4  The room was clean and spacious. The reception...  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Review_ID     1000 non-null   int64 
 1   Hotel_Name    1000 non-null   object
 2   Rating        1000 non-null   int64 
 3   Guest_Review  1000 non-null   object
dtypes: int64(2), object(2)
memory usage: 31.4+ KB
None
Dataset Shape: (

In [2]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [3]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):

    text = str(text).lower()

    text = re.sub(r'[^a-zA-Z ]', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

df["Clean_Review"] = df["Guest_Review"].apply(preprocess)

print(df[["Guest_Review","Clean_Review"]].head())

                                        Guest_Review  \
0  The room was small but comfortable. The staff ...   
1  The room was small but comfortable. The staff ...   
2  The air conditioning worked perfectly. The sta...   
3  The bed was very comfortable. The housekeeping...   
4  The room was clean and spacious. The reception...   

                                        Clean_Review  
0  room small comfortable staff rude food cold gy...  
1  room small comfortable staff friendly restaura...  
2  air conditioning worked perfectly staff friend...  
3  bed comfortable housekeeping attentive restaur...  
4  room clean spacious reception helpful food ave...  


In [4]:
sia = SentimentIntensityAnalyzer()

def sentiment(review):

    score = sia.polarity_scores(review)["compound"]

    if score >= 0.05:
        return "Positive"

    elif score <= -0.05:
        return "Negative"

    else:
        return "Neutral"

df["Sentiment"] = df["Guest_Review"].apply(sentiment)

print(df[["Guest_Review","Sentiment"]].head())

                                        Guest_Review Sentiment
0  The room was small but comfortable. The staff ...  Positive
1  The room was small but comfortable. The staff ...  Positive
2  The air conditioning worked perfectly. The sta...  Positive
3  The bed was very comfortable. The housekeeping...  Positive
4  The room was clean and spacious. The reception...  Positive


In [5]:
def service_issue(review):

    review = review.lower()

    if "room" in review:
        return "Room"

    elif "staff" in review or "reception" in review:
        return "Staff"

    elif "food" in review or "restaurant" in review or "breakfast" in review:
        return "Food"

    elif "clean" in review or "dirty" in review or "housekeeping" in review:
        return "Cleanliness"

    elif "wifi" in review or "pool" in review or "gym" in review or "spa" in review:
        return "Amenities"

    else:
        return "Other"

df["Service_Issue"] = df["Guest_Review"].apply(service_issue)

print(df[["Guest_Review","Service_Issue"]].head())

                                        Guest_Review Service_Issue
0  The room was small but comfortable. The staff ...          Room
1  The room was small but comfortable. The staff ...          Room
2  The air conditioning worked perfectly. The sta...          Room
3  The bed was very comfortable. The housekeeping...          Food
4  The room was clean and spacious. The reception...          Room


In [8]:
print("Customer Satisfaction Report\n")

print("Total Reviews :", len(df))

print("\nSentiment Distribution")
print(df["Sentiment"].value_counts())

print("\nService Issues")
print(df["Service_Issue"].value_counts())

positive = (df["Sentiment"] == "Positive").mean() * 100
negative = (df["Sentiment"] == "Negative").mean() * 100
neutral = (df["Sentiment"] == "Neutral").mean() * 100

print("\nPositive Reviews :", round(positive,2), "%")
print("Negative Reviews :", round(negative,2), "%")
print("Neutral Reviews :", round(neutral,2), "%")

Customer Satisfaction Report

Total Reviews : 1000

Sentiment Distribution
Sentiment
Positive    911
Negative     69
Neutral      20
Name: count, dtype: int64

Service Issues
Service_Issue
Room           729
Staff          173
Food            87
Cleanliness      8
Amenities        2
Other            1
Name: count, dtype: int64

Positive Reviews : 91.1 %
Negative Reviews : 6.9 %
Neutral Reviews : 2.0 %


In [7]:
sample_review = "The room was clean and spacious. Staff were friendly and breakfast was delicious."

clean_review = preprocess(sample_review)

print("Clean Review:", clean_review)
print("Sentiment:", sentiment(sample_review))
print("Service Issue:", service_issue(sample_review))

Clean Review: room clean spacious staff friendly breakfast delicious
Sentiment: Positive
Service Issue: Room
